# Generate Synthetic FLAIR Images — v3 Epoch 90
Load the best v3 checkpoint (epoch 90) and generate synthetic FLAIR images with:
- Sequential naming (`synthetic_0001.png`, `synthetic_0002.png`, ...)
- EMA weights for sharpest quality
- Tumor size filter (skip >8% or <50px tumors)
- Quality filter (skip noise/bad samples)
- Auto-save to both local and Google Drive

## 1. Setup & Imports

In [ ]:
import os, torch, glob
import numpy as np
import tifffile
from PIL import Image
from src.med_ddpm_v3 import ConditionalDDPM, CONFIG

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = CONFIG["synthetic_dir"]
drive_output = "/content/drive/MyDrive/mask-to-mri/outputs_v3/synthetic"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(drive_output, exist_ok=True)

print(f"Device: {device}")
print(f"Output: {output_dir}")
print(f"Drive:  {drive_output}")

## 2. Load Checkpoint (EMA Weights)

In [ ]:
# Find the latest v3 checkpoint or use epoch 90
drive_ckpts = glob.glob("/content/drive/MyDrive/mask-to-mri/outputs_v3/checkpoints/checkpoint_v3_epoch_*.pt")
if drive_ckpts:
    ckpt_path = max(drive_ckpts)
    print(f"Found latest checkpoint: {os.path.basename(ckpt_path)}")
else:
    ckpt_path = "/content/drive/MyDrive/mask-to-mri/outputs_v3/checkpoints/checkpoint_v3_epoch_90.pt"
    print(f"Using epoch 90 checkpoint")

if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

print(f"Loading: {os.path.basename(ckpt_path)}")

model = ConditionalDDPM(CONFIG).to(device)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

# Always prefer EMA weights — they produce the sharpest samples
if "ema_state_dict" in ckpt:
    model.load_state_dict(ckpt["ema_state_dict"])
    print("✅ Loaded EMA weights")
else:
    model.load_state_dict(ckpt["model_state_dict"])
    print("⚠️  No EMA, using raw weights")

model.eval()
print("Model ready.")

## 3. Load Training Masks

In [ ]:
from src.dataset import get_patient_file_list, patient_level_split

patient_data = get_patient_file_list(CONFIG["raw_dir"])
splits = patient_level_split(patient_data, seed=CONFIG["seed"])
train_pairs = splits["train"]
print(f"Loaded {len(train_pairs)} training mask pairs")

## 4. Generate Images

In [ ]:
count = 0
skipped_noise = 0
skipped_tumor = 0

# Change this to generate more (or fewer) images
TARGET_IMAGES = 4

for img_path, mask_path in train_pairs:
    if count >= TARGET_IMAGES:
        break

    m = tifffile.imread(mask_path)
    if (m > 0).sum() == 0:
        continue

    stem = os.path.basename(mask_path).replace("_mask.tif", "")

    # ── Tumor size filter ──
    tumor_pixels = (m > 0).sum()
    total_pixels = m.shape[0] * m.shape[1]
    tumor_ratio = tumor_pixels / total_pixels

    if tumor_ratio > 0.08 or tumor_pixels < 50:
        print(f"⚠️  Skipping {stem} — tumor ratio={tumor_ratio:.3f}, pixels={tumor_pixels}")
        skipped_tumor += 1
        continue

    # ── Generate ──
    m = (m > 0).astype(np.uint8) * 255
    m_norm = (m.astype(np.float32) / 127.5) - 1.0
    mask_t = torch.from_numpy(m_norm).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        fake = model.sample(mask_t, ddim_steps=250)

    # ── Quality filter ──
    fake_std = fake.std().item()
    fake_mean = fake[0, 0].cpu().numpy().mean()

    if fake_std < 0.15 or fake_mean > -0.3:
        print(f"⚠️  Skipping {stem} — bad sample (std={fake_std:.3f}, mean={fake_mean:.3f})")
        skipped_noise += 1
        continue

    fake_np = ((fake[0, 0].cpu().numpy() + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

    # Increment counter for naming
    count += 1
    new_name = f"synthetic_{count:04d}"

    # Save to local and Drive
    for dest in [output_dir, drive_output]:
        Image.fromarray(fake_np, mode="L").save(os.path.join(dest, f"{new_name}.png"))
        Image.fromarray(m).save(os.path.join(dest, f"{new_name}_mask.png"))

    print(f"✅ {new_name} (was {stem}) — std={fake_std:.3f}, mean={fake_mean:.3f}, tumor={tumor_ratio:.1%}")

print(f"\n{'='*60}")
print(f"Done. {count} images saved, {skipped_tumor} skipped (tumor size), {skipped_noise} skipped (noise)")
print(f"Saved to: {drive_output}")
print(f"{'='*60}")

## 5. Preview Generated Images

In [ ]:
import matplotlib.pyplot as plt

# Find the generated images
synthetic_files = sorted(glob.glob(f"{drive_output}/synthetic_*.png"))
# Filter to only main images, not masks
synthetic_files = [f for f in synthetic_files if not f.endswith('_mask.png')]

n = min(4, len(synthetic_files))
if n == 0:
    print("No generated images found.")
else:
    fig, axes = plt.subplots(n, 2, figsize=(10, 5*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n):
        fake_img = np.array(Image.open(synthetic_files[i]))
        mask_img_path = synthetic_files[i].replace('.png', '_mask.png')
        if os.path.exists(mask_img_path):
            mask_img = np.array(Image.open(mask_img_path))
        else:
            mask_img = np.zeros_like(fake_img)
        
        axes[i, 0].imshow(mask_img, cmap='gray')
        axes[i, 0].set_title('Input Mask')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(fake_img, cmap='gray')
        axes[i, 1].set_title('Generated FLAIR')
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    plt.show()